# Linear Slim — median impute (S1 Equities)

Pooled **Ridge** on engineered features after an editable **`FEATURE_REMOVE`** exclude-list.
Target = within-date pct rank of `fwd_ret_5` on week-start rows.

Same chronological IS → train / embargo / val → sealed holdout as `02_training_linear_baseline`.
NaN policy: CS-rank features (leave missing), then fill with **train-window medians**
(fit on train ranks only; applied to val / holdout / full panel).

## Ridge vs ordinary least squares (OLS)

OLS fits coefficients by minimizing squared prediction error only. With several
cross-sectionally ranked factors, those coefficients can become large and unstable
when features are correlated.

**Ridge** is the same linear model with an L2 penalty: it minimizes
`||y − Xβ||² + α||β||²`. The penalty shrinks coefficients toward zero. This notebook
uses Ridge and chooses `α` on the sealed validation window by mean date IC.


## 0. Imports & Config


In [1]:
import os
import sys
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.linear_model import Ridge

warnings.filterwarnings("ignore")

ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(ROOT, "01_data", "ingestion")):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        break
    ROOT = parent
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from models.s1_equities.training_common import (
    LABEL_COL,
    TARGET_COL,
    add_cs_pct_target,
    attach_scores,
    chronological_is_split,
    cs_rank_features,
    default_paths,
    drop_nonfinite_labels,
    ic_segment_table,
    mean_date_ic,
    prepare_s1_week_panel,
)

PATHS = default_paths(ROOT)
FEATURES_PATH = PATHS["features"]
TRAIN_PANEL_PATH = PATHS["train_panel"]
PRED_PATH = os.path.join(PATHS["model_dir"], "s1_linear_slim_is_predictions.parquet")
os.makedirs(PATHS["model_dir"], exist_ok=True)

VAL_FRAC = 0.15
EMBARGO_WEEKS = 1
ALPHA_GRID = [0.1, 1.0, 10.0, 100.0]
RANDOM_SEED = 42

# Editable remove-list (screening names mapped to engineered columns where needed).
# Comment / uncomment lines to add or remove factors from the *exclude* set.
# Names missing from the panel are skipped with a warning.
FEATURE_REMOVE = [
    "log_mcap",
    "val_mom_dist_252_21",
    "downside_beta_42",
    "short_flow_ratio",  # screening: short_flow_abnormal_1_120 (not in matrix)
    "filing_clock_expected_until",  # screening: filing_clock_since
    "earnings_yield",
    "smart_beta_mom_126",
    "gdelt_abnormal_attention_5_60",
    "smart_beta_smb_84",
    # "val_mom_interact_252_21",  # not in engineered matrix
    "val_mom_resid_126_252_10",
    "upside_beta_42",
    "gdelt_attention_5",
    "obv_mom_soft_126_21_20",
    "smart_residual_mom_189_42",  # screening: smart_residual_mom_189_21
    # "net_beta_spread_63",  # not in engineered matrix
    "val_roc_pe_252",
    "rel_upside_beta_63",
    "gross_profitability",
    "near_52w_ratio_252_raw",
    # "near_52w_log_drawdown_252_raw",  # not in engineered matrix
]

print(f"ROOT={ROOT}")
print(f"FEATURES_PATH={FEATURES_PATH}")
print(f"PRED_PATH={PRED_PATH}")
print(f"FEATURE_REMOVE requested ({len(FEATURE_REMOVE)}): {FEATURE_REMOVE}")

ROOT=c:\Users\User\Desktop\ML Algorithmic Trading\Portfolio 26\trading_portfolio
FEATURES_PATH=c:\Users\User\Desktop\ML Algorithmic Trading\Portfolio 26\trading_portfolio\01_data\data_files\s1_equities\s1_engineered_features.parquet
PRED_PATH=c:\Users\User\Desktop\ML Algorithmic Trading\Portfolio 26\trading_portfolio\03_models\s1_equities\model_artifacts\s1_linear_slim_is_predictions.parquet
FEATURE_REMOVE requested (18): ['log_mcap', 'val_mom_dist_252_21', 'downside_beta_42', 'short_flow_ratio', 'filing_clock_expected_until', 'earnings_yield', 'smart_beta_mom_126', 'gdelt_abnormal_attention_5_60', 'smart_beta_smb_84', 'val_mom_resid_126_252_10', 'upside_beta_42', 'gdelt_attention_5', 'obv_mom_soft_126_21_20', 'smart_residual_mom_189_42', 'val_roc_pe_252', 'rel_upside_beta_63', 'gross_profitability', 'near_52w_ratio_252_raw']


## 1. Load week-start panel

Uses `prepare_s1_week_panel`, then drops `FEATURE_REMOVE` names from the full feature set.


In [2]:
df, all_feats, _prices = prepare_s1_week_panel(FEATURES_PATH, TRAIN_PANEL_PATH)

missing = [c for c in FEATURE_REMOVE if c not in df.columns]
remove_resolved = [c for c in FEATURE_REMOVE if c in df.columns]
if missing:
    print(f"WARNING: {len(missing)} remove-list name(s) not in panel (skipped): {missing}")

FEATURE_COLS = [c for c in all_feats if c not in set(remove_resolved)]
print(f"FEATURE_REMOVE resolved ({len(remove_resolved)}): {remove_resolved}")
print(f"FEATURE_COLS kept ({len(FEATURE_COLS)}): {FEATURE_COLS}")
print(f"Week-start rows: {df.shape}  unique weeks={df['date'].nunique():,}")
print(
    f"IS weeks={df.loc[df['is_research_is'], 'date'].nunique():,}  "
    f"holdout weeks={df.loc[~df['is_research_is'], 'date'].nunique():,}"
)
df.head()

FEATURE_REMOVE resolved (18): ['log_mcap', 'val_mom_dist_252_21', 'downside_beta_42', 'short_flow_ratio', 'filing_clock_expected_until', 'earnings_yield', 'smart_beta_mom_126', 'gdelt_abnormal_attention_5_60', 'smart_beta_smb_84', 'val_mom_resid_126_252_10', 'upside_beta_42', 'gdelt_attention_5', 'obv_mom_soft_126_21_20', 'smart_residual_mom_189_42', 'val_roc_pe_252', 'rel_upside_beta_63', 'gross_profitability', 'near_52w_ratio_252_raw']
FEATURE_COLS kept (9): ['raw_momentum_252_5', 'rel_downside_beta_252', 'smart_beta_hml_252', 'size_mom_126', 'val_roc_pb_252', 'market_corr', 'beta_mkt_interact', 'abnormal_volume', 'gdelt_tone_x_attention_21']
Week-start rows: (85279, 39)  unique weeks=865
IS weeks=422  holdout weeks=443


,date,ticker,feature_date,open,high,low,close,volume,fwd_ret_1,fwd_ret_5,...,gross_profitability,filing_clock_expected_until,short_flow_ratio,market_corr,beta_mkt_interact,abnormal_volume,gdelt_tone_x_attention_21,gdelt_attention_5,gdelt_abnormal_attention_5_60,is_research_is
0,2010-01-05,AAPL,2010-01-04,6.424143,6.421146,6.357683,6.406478,493729600.0,-0.001025,-0.025210,...,NaN,27.0,0.439729,NaN,NaN,NaN,NaN,NaN,NaN,False
1,2010-01-05,ABT,2010-01-04,18.082116,18.111994,17.899535,18.078796,10829095.0,-0.009730,0.013402,...,NaN,31.0,0.255644,NaN,NaN,NaN,NaN,NaN,NaN,False
2,2010-01-05,ADBE,2010-01-04,37.040001,37.299999,36.650002,37.090000,4710200.0,0.007829,-0.024298,...,NaN,1.0,0.383420,NaN,NaN,NaN,NaN,NaN,NaN,False
3,2010-01-05,AET,2010-01-04,29.889494,30.016532,28.918587,29.943939,5671979.0,-0.013358,-0.009411,...,NaN,NaN,0.295322,NaN,NaN,NaN,NaN,NaN,NaN,False
4,2010-01-05,AIG,2010-01-04,18.609565,18.957173,18.255744,18.553696,7750900.0,-0.021014,-0.013009,...,NaN,4.0,0.417434,NaN,NaN,NaN,NaN,NaN,NaN,False


## 2. CS-rank target


In [3]:
df = add_cs_pct_target(df)
n_before = len(df)
df = drop_nonfinite_labels(df, [LABEL_COL, TARGET_COL])
print(
    f"Rows after finite label/target: {df.shape}  "
    f"(retained {100 * len(df) / max(n_before, 1):.2f}%)"
)
print(
    f"Target mean={df[TARGET_COL].mean():.4f}  std={df[TARGET_COL].std():.4f}"
)

Rows after finite label/target: (85179, 40)  (retained 99.88%)
Target mean=0.5051  std=0.2887


## 3. Train / Val / Holdout Split


In [4]:
split = chronological_is_split(df, val_frac=VAL_FRAC, embargo_weeks=EMBARGO_WEEKS)
train_dates, val_dates = split.train_dates, split.val_dates
is_end = split.is_end
train_df, val_df, holdout_df = split.train_df, split.val_df, split.holdout_df

for label, part in (
    ("train", train_df),
    ("val", val_df),
    ("holdout", holdout_df),
):
    print(f"{label}: rows={len(part):,}  weeks={part['date'].nunique()}")

print(
    f"Train:   {train_df.shape}  {train_dates.min().date()} -> {train_dates.max().date()}  "
    f"({len(train_dates)} weeks)"
)
print(f"Embargo: {list(split.embargo_dates.date)}  ({len(split.embargo_dates)} weeks)")
print(
    f"Val:     {val_df.shape}  {val_dates.min().date()} -> {val_dates.max().date()}  "
    f"({len(val_dates)} weeks)"
)
print(
    f"Holdout: {holdout_df.shape}  "
    f"{holdout_df['date'].min().date() if len(holdout_df) else 'n/a'} -> "
    f"{holdout_df['date'].max().date() if len(holdout_df) else 'n/a'}  "
    f"({holdout_df['date'].nunique()} weeks)"
)

train: rows=35,314  weeks=358
val: rows=6,111  weeks=63
holdout: rows=17,557  weeks=181
Train:   (35314, 40)  2015-01-05 -> 2021-11-08  (358 weeks)
Embargo: [datetime.date(2021, 11, 15)]  (1 weeks)
Val:     (6111, 40)  2021-11-22 -> 2023-01-30  (63 weeks)
Holdout: (17557, 40)  2023-02-06 -> 2026-07-20  (181 weeks)


## 4. CS-rank features + train-median impute

Within-date CS pct-rank each feature (NaNs left as-is).
Medians are computed on **train** ranks only, then used to fill NaNs everywhere.


In [5]:
X_ranked = cs_rank_features(df, FEATURE_COLS, fill_value=None)

nan_share = X_ranked.isna().mean().sort_values(ascending=False)
print("Per-feature NaN share after CS-rank (before impute):")
display((100 * nan_share).rename("nan_pct").to_frame().round(2).head(20))

train_medians = X_ranked.loc[train_df.index].median(axis=0)
n_all_nan = int(train_medians.isna().sum())
if n_all_nan:
    # Feature entirely missing on train → fill remaining with 0.5 (neutral rank)
    train_medians = train_medians.fillna(0.5)
    print(
        f"WARNING: {n_all_nan} feature(s) had no finite train values; "
        "used neutral rank 0.5 for those medians."
    )

n_nan_before = int(X_ranked.isna().sum().sum())
X = X_ranked.fillna(train_medians)
n_nan_after = int(X.isna().sum().sum())
print(
    f"Train-median impute: NaN cells before={n_nan_before:,}  after={n_nan_after:,}"
)
display(train_medians.sort_values().rename("train_median").to_frame().head(15))

Per-feature NaN share after CS-rank (before impute):


,nan_pct
gdelt_tone_x_attention_21,42.90
val_roc_pb_252,28.95
smart_beta_hml_252,26.30
size_mom_126,15.24
raw_momentum_252_5,6.22
rel_downside_beta_252,6.22
market_corr,6.22
beta_mkt_interact,6.22
abnormal_volume,1.64


Train-median impute: NaN cells before=119,187  after=0


,train_median
gdelt_tone_x_attention_21,0.500000
raw_momentum_252_5,0.505155
smart_beta_hml_252,0.505155
rel_downside_beta_252,0.505155
abnormal_volume,0.505155
beta_mkt_interact,0.505155
market_corr,0.505155
size_mom_126,0.505747
val_roc_pb_252,0.506667


## 5. Fit Ridge

`α` selected on val by mean date IC.


In [6]:
X_train = X.loc[train_df.index]
y_train = train_df[TARGET_COL]
X_val = X.loc[val_df.index]

search_rows = []
for alpha in ALPHA_GRID:
    model = Ridge(alpha=alpha, random_state=RANDOM_SEED)
    model.fit(X_train, y_train)
    pred_va = attach_scores(val_df, model.predict(X_val))
    sm = mean_date_ic(pred_va)
    search_rows.append(
        {
            "alpha": alpha,
            "val_mean_ic": sm["mean_ic"],
            "val_icir": sm["icir"],
            "n_dates": sm["n"],
        }
    )
    print(
        f"alpha={alpha:<6g}  val_ic={sm['mean_ic']:.4f}  "
        f"ICIR={sm['icir']:.3f}  n={sm['n']}"
    )

search_df = (
    pd.DataFrame(search_rows)
    .sort_values(["val_mean_ic", "val_icir"], ascending=False)
    .reset_index(drop=True)
)
display(search_df)

BEST_ALPHA = float(search_df.iloc[0]["alpha"])
print(f"BEST_ALPHA={BEST_ALPHA}  (val mean IC={search_df.iloc[0]['val_mean_ic']:.4f})")

alpha=0.1     val_ic=-0.0447  ICIR=-0.138  n=63
alpha=1       val_ic=-0.0447  ICIR=-0.138  n=63
alpha=10      val_ic=-0.0448  ICIR=-0.139  n=63
alpha=100     val_ic=-0.0453  ICIR=-0.140  n=63


,alpha,val_mean_ic,val_icir,n_dates
0,1.0,-0.044663,-0.138040,63
1,0.1,-0.044696,-0.138152,63
2,10.0,-0.044819,-0.138513,63
3,100.0,-0.045251,-0.140097,63


BEST_ALPHA=1.0  (val mean IC=-0.0447)


## 6. Final IS fit & predictions


In [7]:
model = Ridge(alpha=BEST_ALPHA, random_state=RANDOM_SEED)
model.fit(X_train, y_train)

preds = attach_scores(df, model.predict(X))
if "feature_date" in df.columns:
    preds = preds.merge(
        df[["date", "ticker", "feature_date"]],
        on=["date", "ticker"],
        how="left",
    )
preds.to_parquet(PRED_PATH, index=False)
print(f"Saved predictions: {PRED_PATH}")
print(
    f"  rows={len(preds):,}  IS={preds['is_research_is'].sum():,}  "
    f"non-IS={(~preds['is_research_is']).sum():,}"
)

coef = (
    pd.Series(model.coef_, index=FEATURE_COLS)
    .sort_values(key=np.abs, ascending=False)
    .rename("coef")
)
display(coef.head(20).to_frame())

Saved predictions: c:\Users\User\Desktop\ML Algorithmic Trading\Portfolio 26\trading_portfolio\03_models\s1_equities\model_artifacts\s1_linear_slim_is_predictions.parquet
  rows=85,179  IS=41,522  non-IS=43,657


,coef
smart_beta_hml_252,-0.045271
market_corr,0.016273
gdelt_tone_x_attention_21,0.012141
val_roc_pb_252,-0.011916
size_mom_126,0.011644
beta_mkt_interact,0.010600
abnormal_volume,-0.001461
raw_momentum_252_5,0.000927
rel_downside_beta_252,0.000804


## 7. IC summary


In [8]:
ic_compare = ic_segment_table(preds, train_dates, val_dates, is_end)
display(ic_compare)

ho = ic_compare.loc["OS holdout"]
tv = ic_compare.loc["IS train+val"]
print(
    f"IS train+val IC={tv['mean_ic']:.4f} ICIR={tv['icir']:.3f}  |  "
    f"OS holdout IC={ho['mean_ic']:.4f} ICIR={ho['icir']:.3f}"
)

,mean_ic,icir,n_dates
segment,,,
IS train,0.051528,0.219843,358
IS val,-0.044663,-0.138040,63
IS train+val,0.037133,0.147558,421
OS holdout,0.023879,0.086471,181


IS train+val IC=0.0371 ICIR=0.148  |  OS holdout IC=0.0239 ICIR=0.086
